# Aula 13 — Feedback humano: o fiscal que risca e reescreve

**Objetivo:** sair do carimbo binário (retoma/não retoma) para a edição de
verdade: corrigir o pedido parado, anexar contexto e assinar a intervenção.

**Pré-requisitos:** Aula 12 (pausa/inspeciona/retoma) e Aula 08 (`add_messages`).

**Como usar:** rode em ordem. Autocontido (1 arquivo no Colab). Comece pela
CÉLULA 0. Cole sua chave na CÉLULA 1 (só precisa no Colab/sem .env).
Com `GROQ_API_KEY` usa Groq real; sem chave, Fake offline.

In [ ]:
# CÉLULA 0 — instalar dependências
# No Colab esta célula é necessária. Localmente (uv sync feito) pode pular.
%pip install -q langgraph langchain-groq python-dotenv
print('dependências prontas.')

In [ ]:
# CÉLULA 1 — chave do Groq (opcional: sem ela, tudo roda no Fake offline)
import getpass
import os
from dotenv import load_dotenv

load_dotenv()  # lê .env se existir (uso local); no Colab não há .env

if not os.getenv('GROQ_API_KEY'):
    try:
        os.environ['GROQ_API_KEY'] = getpass.getpass('GROQ_API_KEY (digitação oculta): ')
    except Exception:
        print('sem entrada interativa -> seguirei em modo Fake offline.')

print('chave presente:', bool(os.getenv('GROQ_API_KEY')))
# NUNCA dê print na chave. Verificamos só presença (True/False).

## 1. Analogia: o fiscal com caneta vermelha

Na Aula 12 o fiscal só carimbava. Agora ele tem caneta vermelha:

- **Risca e reescreve o pedido** ('não é soma, é multiplicação por 10') — o
  caixa executa a versão corrigida, e a original some.
- **Rabisa contexto na comanda** ('cliente com pressa') — o pedido continua
  valendo, com uma linha a mais.
- **Assina a intervenção** ('fiscal: Ana') — no histórico fica claro que aquela
  linha foi humana, não do atendente nem do caixa.

| Caneta vermelha | No LangGraph | O que é de verdade |
|---|---|---|
| reescrever o pedido | **`update_state` com MESMO `id`** | `add_messages` sobrescreve |
| rabiscar contexto | **`update_state` sem `id`** | `add_messages` anexa |
| assinar 'fiscal: Ana' | **`as_node='feedback_humano'`** | atribui a edição a um nó (que precisa existir) |

In [ ]:
# CÉLULA 2 — base: ReAct pausado + fábrica de LLM fresco por grafo
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langgraph.graph import MessagesState, StateGraph, START
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver

SISTEMA = 'Você é um assistente que faz contas. Responda em português, de forma curta.'
MODELO = 'openai/gpt-oss-20b'

def add(a: int, b: int) -> int:
    """Soma a e b."""
    return a + b

def multiply(a: int, b: int) -> int:
    """Multiplica a por b."""
    return a * b

TOOLS = [add, multiply]

def novo_llm():
    # UM LLM fresco por grafo (regra das Aulas 06/11/12 contra sequência embaralhada).
    if os.getenv('GROQ_API_KEY'):
        try:
            from langchain_groq import ChatGroq
        except ImportError:
            print('AVISO: langchain_groq ausente -> Fake offline.')
        else:
            return ChatGroq(model=MODELO, temperature=0,
                            reasoning_effort='low').bind_tools(TOOLS)
    from langchain_core.language_models.fake_chat_models import FakeMessagesListChatModel
    print('AVISO: modo Fake offline.')
    return FakeMessagesListChatModel(responses=[
        AIMessage(content='', id='pedido-1', tool_calls=[
            {'name': 'add', 'args': {'a': 3, 'b': 4}, 'id': 'c1', 'type': 'tool_call'}]),
        AIMessage(content='Fechado.', tool_calls=[]),
    ])

def montar(com_feedback=False, interrupt_em=None):
    # interrupt_em: onde pausar (padrão: antes das tools). Para o placeholder,
    # pausamos antes dele: interrupt_em=['feedback_humano'].
    pausa = list(interrupt_em) if interrupt_em else ['tools']
    llm = novo_llm()
    def assistant(state: MessagesState) -> dict:
        return {'messages': [llm.invoke(
            [SystemMessage(content=SISTEMA)] + state['messages'])]}
    b = StateGraph(MessagesState)
    if com_feedback:
        b.add_node('feedback_humano', lambda state: {})  # placeholder: só dá nome
        b.add_edge(START, 'feedback_humano')
        b.add_edge('feedback_humano', 'assistant')
    else:
        b.add_edge(START, 'assistant')
    b.add_node('assistant', assistant)
    b.add_node('tools', ToolNode(TOOLS))
    b.add_conditional_edges('assistant', tools_condition)
    b.add_edge('tools', 'assistant')
    return b.compile(checkpointer=MemorySaver(), interrupt_before=pausa)

print('fábrica pronta: cada montar() tem seu LLM fresco.')

## 2. Conceito: `update_state` escreve no parado

- `graph.update_state(cfg, {'messages': [...]})` edita o estado **do checkpoint
  atual** (o parado na pausa). Não executa nada — só reescreve o retrato.
- Como a chave é `messages` com `add_messages`, valem as regras da Aula 08:
  **mesmo `id` sobrescreve** (corrige), **sem `id`** (ou id novo) **anexa**.
- `as_node='feedback_humano'` carimba *quem* editou no histórico. Sem ele, a
  edição entra como se fosse do último nó executado — historicamente confuso.
- Depois de editar, `stream(None, cfg)` retoma **a partir do estado editado**:
  o fiscal editou a bancada, o caixa executa a versão nova.

In [ ]:
# CÉLULA 3 — corrige o pedido: mesmo id, outra tool (rode e confira o 30)
grafo = montar()
cfg = {'configurable': {'thread_id': 'corrige'}}
list(grafo.stream({'messages': [HumanMessage(content='Some 3 e 4')]}, cfg))
pedido = grafo.get_state(cfg).values['messages'][-1]
print('pedido parado:', pedido.tool_calls[0]['name'], pedido.tool_calls[0]['args'])

corrigido = AIMessage(content='', id=pedido.id, tool_calls=[
    {'name': 'multiply', 'args': {'a': 3, 'b': 10}, 'id': 'c1', 'type': 'tool_call'}])
grafo.update_state(cfg, {'messages': [corrigido]})   # MESMO id -> sobrescreve
list(grafo.stream(None, cfg))
recibos = [m.content for m in grafo.get_state(cfg).values['messages']
           if type(m).__name__ == 'ToolMessage']
print('recibos:', recibos)
if os.getenv('GROQ_API_KEY'):
    print('(com Groq real, o pedido parado depende do modelo; o mecanismo é o mesmo:)')
    print(' mesmo id = corrige, id novo = anexa.)')
else:
    assert recibos == ['30'], recibos
    print('OK — executou multiply(3,10), não add(3,4).')

## 3. Dissecando a correção (por que o `id` decide tudo)

- `pedido.id` copiado para o `corrigido`: o reducer casa os dois e **troca**.
  Sem copiar (id novo ou ausente), o corrigido **anexaria** e o pedido original
  continuaria valendo — o caixa executaria o errado.
- `tool_calls[0]['id']` (`c1`) foi mantido: é o grampo pedido-recibo (Aula 04).
  Trocar o `name`/`args` mantendo o `id` do pedido diz 'mesmo pedido, conta nova'.
- `stream(None, cfg)` após editar: retoma do estado **editado**. O recibo `30`
  prova que o executado foi o corrigido.

In [ ]:
# CÉLULA 4 — anexa contexto: sem id, soma (rode e conte)
g2 = montar()
cfg2 = {'configurable': {'thread_id': 'anexa'}}
list(g2.stream({'messages': [HumanMessage(content='Some 3 e 4')]}, cfg2))
antes = len(g2.get_state(cfg2).values['messages'])
g2.update_state(cfg2, {'messages': [HumanMessage(content='urgente: preciso hoje')]})
depois = len(g2.get_state(cfg2).values['messages'])
print(f'antes: {antes} msgs | depois: {depois} msgs')
assert depois == antes + 1
print('OK — anexar soma; só o MESMO id substitui.')

## 4. Variações que quebram (todas executadas e verificadas)

### 4a. `as_node` com nome inexistente: `InvalidUpdateError`

Assinar como nó que não está no grafo morre na hora — a assinatura precisa de
um 'carimbo' registrado. É para isso que existe o **nó placeholder**:
uma função que devolve `{}` (nunca faz nada de verdade), só para dar nome
às intervenções humanas no histórico.

In [ ]:
# CÉLULA 5 — as_node fantasma (100% offline na lógica)
g3 = montar()
cfg3 = {'configurable': {'thread_id': 'fantasma'}}
list(g3.stream({'messages': [HumanMessage(content='Some 3 e 4')]}, cfg3))
try:
    g3.update_state(cfg3, {'messages': [HumanMessage(content='oi')]}, as_node='fantasma')
    print('assinou?! (não deveria)')
except Exception as erro:
    print(type(erro).__name__ + ':', str(erro)[:60])
    print('OK — assinatura exige carimbo registrado (próxima célula).')

### 4b. Placeholder + `as_node`: a edição entra assinada

O padrão completo: pausa-se **antes** do nó placeholder, o humano escreve
`as_node='feedback_humano'`, e o fluxo segue placeholder → assistant → ...
normalmente. No histórico, a linha humana aparece no checkpoint do placeholder.

In [ ]:
# CÉLULA 6 — feedback assinado (rode e leia o histórico)
g6 = montar(com_feedback=True, interrupt_em=["feedback_humano"])
cfg6 = {'configurable': {'thread_id': 'assinado'}}
list(g6.stream({'messages': [HumanMessage(content='Some 3 e 4')]}, cfg6))
print('pausado em:', g6.get_state(cfg6).next)
g6.update_state(cfg6,
                {'messages': [HumanMessage(content='na verdade quero o dobro: some 7 e 7')]},
                as_node='feedback_humano')
list(g6.stream(None, cfg6))
msgs = g6.get_state(cfg6).values['messages']
print('histórico:', [(type(m).__name__, m.content) for m in msgs])
assert msgs[1].content == 'na verdade quero o dobro: some 7 e 7'
print('OK — a linha humana entrou assinada como feedback_humano.')

## 5. Erros clássicos

1. **Corrigir sem copiar o `id`.** Id novo = anexa; o pedido original continua
   valendo e o caixa executa o errado. Correção sempre carrega o `id` velho.
2. **Trocar o `id` do `tool_call` (`c1`) sem querer.** O grampo pedido-recibo
   mora nele; `id` novo no `tool_calls` confunde o `ToolNode` na volta.
3. **`as_node` sem placeholder.** Assinatura exige carimbo registrado
   (CÉLULA 5). Sem o nó, `InvalidUpdateError`.
4. **Editar e esquecer de retomar.** `update_state` não executa nada — sem o
   `stream(None)`, a thread fica parada com a edição guardada.
5. **Editar o checkpoint errado.** `update_state` escreve no checkpoint **atual**
   da thread. Para reescrever o passado, use o `checkpoint_id` antigo (Aula 15).
6. **Reusar o mesmo Fake entre grafos.** Cada `montar()` cria seu LLM fresco —
   sequência compartilhada embaralha (regra das Aulas 06/11/12).

## 6. Exercício guiado — fiscal corrige a conta

Situação: o cliente pediu 'Some 3 e 4', mas o fiscal viu que era 'Some 30 e
40'. Pause, corrija o pedido (mesmo `id`) para `add(30, 40)` e retome.
Valide que o recibo é `'70'`.

In [ ]:
# CÉLULA 7 — exercício guiado (corrige para 30+40)
gex = montar()
cfgex = {'configurable': {'thread_id': 'conta-alta'}}
list(gex.stream({'messages': [HumanMessage(content='Some 3 e 4')]}, cfgex))
ped = gex.get_state(cfgex).values['messages'][-1]
print('pedido parado:', ped.tool_calls[0]['name'], ped.tool_calls[0]['args'])
fix = AIMessage(content='', id=ped.id, tool_calls=[
    {'name': 'add', 'args': {'a': 30, 'b': 40}, 'id': ped.tool_calls[0]['id'],
     'type': 'tool_call'}])
gex.update_state(cfgex, {'messages': [fix]})
list(gex.stream(None, cfgex))
recibos = [m.content for m in gex.get_state(cfgex).values['messages']
           if type(m).__name__ == 'ToolMessage']
print('recibos:', recibos)
if os.getenv('GROQ_API_KEY'):
    print('(com Groq real, o pedido parado varia; o mecanismo é o mesmo.)')
else:
    assert recibos == ['70'], recibos
    print('OK — executou add(30,40), não o original.')

## 7. Desafio — rejeição que redireciona

Em vez de corrigir o pedido, **troque a pergunta**: pause, use `update_state`
para substituir a `HumanMessage` original (mesmo `id` dela!) por 'Quanto é
100 dividido por 4?' e retome.

- Dica: `divide` precisa estar no cardápio — amplie `TOOLS` com ela (Aula 04).
- Valide o recibo `'25.0'` e que a pergunta original sumiu do histórico.
- **Pegadinha:** sobrescrever a `HumanMessage` muda o passado da conversa —
  os checkpoints antigos ainda têm a pergunta velha (Aula 15 mostra como
  navegar entre eles).

## 8. Checklist + próxima aula

**Você entendeu se consegue responder sem olhar:**
- [ ] `update_state` com mesmo `id` vs sem `id`: o que cada um faz?
- [ ] Para que serve o `as_node` — e por que ele exige nó existente?
- [ ] O que o nó placeholder faz (e não faz)?
- [ ] Após editar, o que retoma a execução — e a partir de qual estado?
- [ ] Para corrigir um pedido, quais 2 ids você preserva e por quê?
- [ ] Na sua lanchonete, quem edita o quê: cliente corrige pedido, fiscal corrige o quê?

**Próxima aula (14 — Breakpoints dinâmicos):** pausar por **regra**, não por
lista fixa — `raise NodeInterrupt` condicional dentro do nó (ex: conta > R$ 50).